In [15]:
from anthropic import Anthropic
from pydantic import BaseModel, Field

from jaxn import JSONParserHandler, StreamingJSONParser

In [3]:
client = Anthropic()

In [ ]:
messages = [
    {"role": "user", "content": "Tell me a bad time story about a unicorn."},
]
with client.messages.stream(
    max_tokens=1024,
    model="claude-opus-5-5",
    messages=messages,
) as stream:
    for text in stream.text_stream:
        print(text, end="", flush=True)

# The Very Bad, Extremely Unmagical Day of Sir Glimmerhoof

Once upon a time, in a meadow where the grass smelled like cotton candy, there lived a unicorn named Sir Glimmerhoof. He was having a bad time.

It started when he woke up and discovered his horn had gotten stuck in a tree overnight. Not a little stuck. *Very* stuck. He spent forty-five minutes walking in circles around the trunk until a woodpecker took pity on him and pecked him free. The woodpecker then sent him an invoice.

Next, Sir Glimmerhoof went to the Rainbow Falls for his morning sparkle-bath. But the rainbow was out of order. A sign hung on it that read: *"Closed for Maintenance. Please Enjoy This Gray Puddle Instead."* He enjoyed the gray puddle. He did not enjoy the gray puddle.

At lunchtime, a little girl ran up to him, gasped, and shouted, "A HORSE WITH A TRAFFIC CONE ON ITS HEAD!" Then she tried to feed him a single dry crouton.

He ate the crouton. It was stale.

In the afternoon, he attempted to grant a wish

In [ ]:
class ArticleSection(BaseModel):
    header: str
    text: str = Field(description="text of the section in markdown")


class ArticleResponse(BaseModel):
    title: str
    subtitle: str
    sections: list[ArticleSection]

In [9]:
messages = [
    {"role": "user", "content": "tell me a bad time story about a unicorn"}
]

try:
    with client.messages.stream(
        max_tokens=5000,
        messages=messages,
        model="claude-opus-5-5",
        output_format=ArticleResponse,
    ) as stream:
        for text in stream.text_stream:
            print(text, end="", flush=True)
        
        message = stream.get_final_message()
except Exception as e:
    print(f"Error: {type(e).__name__}: {e}")

{"title":"Pemberton and the Very Bad Day","subtitle":"A bedtime story about a unicorn whose day went wrong, and how it turned out all right","sections":[{"header":"A Morning Gone Wrong","text":"In a meadow where the clover grew as tall as teacups, there lived a small unicorn named Pemberton. Pemberton had a silver mane, hooves that clicked like little bells, and a horn that was supposed to sparkle.\n\n*Supposed to.*\n\nOne morning, Pemberton woke up, gave a big stretch, and bonked his horn right into the low branch of the old apple tree. **THUNK.** An apple fell on his nose. Then another. Then a very surprised squirrel.\n\n\"Oh dear,\" said Pemberton. \"This is not a good start.\""},{"header":"The Sparkle That Wouldn't","text":"Every morning, the unicorns of the meadow gathered by the Glimmering Pond to make their horns sparkle for the sunrise. It was Pemberton's favorite thing.\n\nHe closed his eyes, wished his brightest wish, and...\n\n*fzzzt.*\n\nInstead of a shower of sparkles, his

In [14]:
for block in message.content:
    if block.type == "text":
        story = block.parsed_output
        print('# ' + story.title)
        print(story.subtitle)
        print()

        for section in story.sections:
            print('## ' + section.header)
            print()
            print(section.text)
            print()
            print()

# Pemberton and the Very Bad Day
A bedtime story about a unicorn whose day went wrong, and how it turned out all right

## A Morning Gone Wrong

In a meadow where the clover grew as tall as teacups, there lived a small unicorn named Pemberton. Pemberton had a silver mane, hooves that clicked like little bells, and a horn that was supposed to sparkle.

*Supposed to.*

One morning, Pemberton woke up, gave a big stretch, and bonked his horn right into the low branch of the old apple tree. **THUNK.** An apple fell on his nose. Then another. Then a very surprised squirrel.

"Oh dear," said Pemberton. "This is not a good start."


## The Sparkle That Wouldn't

Every morning, the unicorns of the meadow gathered by the Glimmering Pond to make their horns sparkle for the sunrise. It was Pemberton's favorite thing.

He closed his eyes, wished his brightest wish, and...

*fzzzt.*

Instead of a shower of sparkles, his horn made one sad little puff of gray smoke that smelled faintly of burnt toast.

In [16]:
raw_json = """
{"title":"The Misadventures of Sparkle","subtitle":"A Tale of Trouble","sections":[{"header":"Once Upon a Time","text":"In a magical land far away, there lived a young unicorn named Sparkle."},{"header":"The Problem","text":"Sparkle had a mischievous streak that often led to chaos."}]}
""".strip()

In [ ]:
from typing import Any

class ArticleResponseHandler(JSONParserHandler):
    def on_field_end(self, path: str, field_name: str, value: str, parsed_value: Any = None) -> None:
        if path == '':
            if field_name == 'title':
                print(f'# {value}')
            if field_name == 'subtitle':
                print(value)
        if path == '/sections' and field_name == 'header':
            print(f'\n\n## {value}\n')

    def on_value_chunk(self, path: str, field_name: str, chunk: str) -> None:
        if path == '/sections' and field_name == 'text':
            print(chunk, end='', flush=True)

In [18]:
handler = ArticleResponseHandler()
parser = StreamingJSONParser(handler=handler)

In [19]:
for character in raw_json:
    parser.parse_incremental(character)

# The Misadventures of Sparkle
A Tale of Trouble


## Once Upon a Time

In a magical land far away, there lived a young unicorn named Sparkle.

## The Problem

Sparkle had a mischievous streak that often led to chaos.

In [24]:
def llm_structured_stream(
    user_prompt: str,
    output_type: type[BaseModel],
    parser_handler: JSONParserHandler,
    instructions: str | None =None,
    model: str = "claude-opus-5-5",
):
    messages = []
    if instructions:
        messages.append({
            "role": "assistant",
            "content": instructions
        })

    messages.append({
        "role": "user",
        "content": user_prompt
    })

    parser = StreamingJSONParser(handler=parser_handler)

    with client.messages.stream(
        max_tokens=5000,
        messages=messages,
        model=model,
        output_format=output_type,
    ) as stream:
        for text in stream.text_stream:
            parser.parse_incremental(text)

        response = stream.get_final_message()
    return response

In [25]:
instructions = "your task is to tell the user bad time stories"
user_prompt = "unicorn"

result = llm_structured_stream(
    instructions=instructions,
    user_prompt=user_prompt,
    output_type=ArticleResponse,
    parser_handler=ArticleResponseHandler(),
)

# The Unicorn Who Had a Very Bad Time
A Bad Time Story about Sir Glitterhoof, whose day went wrong at every turn


## Once Upon a Terrible Morning

Once upon a time, in a meadow where the grass was *slightly* too itchy, there lived a unicorn named **Sir Glitterhoof the Magnificent**.

He was not magnificent.

He woke up that morning, stretched his sparkly legs, and immediately stepped in a puddle. It was not a magical puddle. It was just mud. Cold mud. The kind that gets *between* your hooves.

"No matter!" said Sir Glitterhoof. "Today I shall fly over a rainbow!"

Unicorns cannot fly. Nobody had ever told him.

## The Horn Incident

Sir Glitterhoof trotted proudly into the Enchanted Forest, tossing his mane in the wind. He tossed it so hard that his horn went *thwunk* straight into an oak tree.

He was stuck.

A squirrel watched. The squirrel did not help. The squirrel ate an acorn very loudly and then left to tell all the other squirrels.

It took Sir Glitterhoof **four hours** to wi

In [26]:
import json

from gitsource import GithubRepositoryDataReader, chunk_documents
from minsearch import Index


# Read the documentation files from the GitHub repository
reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()


# Parse the files and chunk them into smaller pieces for indexing
parsed_docs = [doc.parse() for doc in files]
chunked_docs = chunk_documents(parsed_docs, size=3000, step=1500)


# Create an index for searching the documentation
index = Index(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"]
)
index.fit(chunked_docs)
print(f"Indexed {len(chunked_docs)} chunks from {len(files)} documents")


# Search function to query the index
def search(query: str):
    results = index.search(
        query=query,
        num_results=5
    )
    return results


instructions = """
You're a documentation assistant. Answer the QUESTION based on the CONTEXT from our documentation.

Use only facts from the CONTEXT when answering.
If the answer isn't in the CONTEXT, say so.
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()


def build_prompt(question: str, search_results: list[dict]):
    context = json.dumps(search_results, indent=2)
    return prompt_template.format(
        question=question,
        context=context
    )

Indexed 382 chunks from 95 documents


In [29]:
from typing import Literal


class RAGResponse(BaseModel):
    """
    This model provides a structured answer with metadata about the response,
    including confidence, categorization, and follow-up suggestions.
    """

    answer: str = Field(description="The main answer to the user's question in markdown")
    found_answer: bool = Field(description="True if relevant information was found in the documentation")
    confidence: float = Field(description="Confidence score from 0.0 to 1.0 indicating how certain the answer is")
    confidence_explanation: str = Field(description="Explanation about the confidence level")
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(description="Suggested follow-up questions the user might want to ask")

In [30]:
class RAGResponseHandler(JSONParserHandler):
    def on_value_chunk(self, path: str, field_name: str, chunk: str) -> None:
        if path == '' and field_name == 'answer':
            print(chunk, end='', flush=True)

    def on_field_end(self, path: str, field_name: str, value: str, parsed_value: Any = None) -> None:
        if path == '' and field_name == 'answer_type':
            print('\nanswer type:', value)

    def on_array_item_end(self, path: str, field_name: str, item: dict[str, Any] = None) -> None:
        if field_name == 'followup_questions':
            print('follow up question:', item)

In [31]:
def rag(query: str):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm_structured_stream(
        instructions=instructions,
        user_prompt=prompt,
        output_type=RAGResponse,
        parser_handler=RAGResponseHandler()
    )

In [32]:
response = rag('llm as a judge')

## LLM as a Judge

The documentation includes a tutorial on evaluating text against custom criteria using an **LLM as the judge**, and then **evaluating the LLM judge itself**. It runs locally with the open-source Evidently Python library. You can optionally upload results to the Evidently Platform at the end.

### Two approaches
- **Reference-based**: Compare new responses against a reference ("ground truth" or approved responses). This is useful for regression testing.
- **Open-ended**: Evaluate responses against custom criteria when no reference is available.

### Requirements
- Basic Python knowledge.
- An OpenAI API key for the LLM evaluator. Other evaluator LLMs can also be used; see the [LLM judge docs](/metrics/customize_llm_judge#change-the-evaluator-llm).
- Jupyter Notebook or Google Colab is recommended.

### Steps

**1. Install and import**
```python
pip install evidently
```
```python
from evidently import Dataset, DataDefinition, Report, BinaryClassification
from evidentl